# LangGraph ganz einfach

**Eine Frage. Ein Modellaufruf. Eine Antwort.**

Start → Modell → Ende

Das Beispiel zeigt einen minimalen LangGraph-Ablauf mit Gemini. Werkzeuge und Verzweigungen folgen im [Rechenagenten](01_langgraph_quickstart.ipynb).

Umgebung und Start: [README](../README.md). Kernel: **LangGraph · langgraph-agent**. Die Zellen von oben nach unten ausführen.

## 1 · Modell und API-Key

Ein eigener [Gemini-API-Key](https://aistudio.google.com/apikey) ist erforderlich.

`getpass` fragt den Schlüssel verdeckt ab. Er gehört nicht als Text in den Code.

Erst `invoke` in Schritt 5 sendet die Frage an Google. Je nach API-Tarif können Kosten entstehen.

In [ ]:
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    api_key=getpass("Gemini-API-Key: "),
    vertexai=False,
)

## 2 · Daten des Ablaufs

Der **State** ist ein Dictionary. Er enthält die Frage und später die Antwort.

`TypedDict` beschreibt seine Struktur.

In [ ]:
from typing import TypedDict

class State(TypedDict):
    frage: str
    antwort: str

## 3 · Ein Knoten

Ein **Knoten** ist eine Funktion. Hier ruft sie Gemini auf und gibt die Antwort als Änderung des State zurück.

Die Definition allein löst noch keinen Aufruf aus.

In [ ]:
def antworten(state: State):
    antwort = model.invoke(state["frage"])
    return {"antwort": antwort.text}

## 4 · Den Weg festlegen

**Kanten** verbinden Start, Modellknoten und Ende.

`compile()` macht den Graphen ausführbar.

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)
graph.add_node("antworten", antworten)
graph.add_edge(START, "antworten")
graph.add_edge("antworten", END)

app = graph.compile()

## 5 · Ausführen

`invoke` startet den Graphen. Hier findet **ein Modellaufruf** statt.

Die Rückgabe enthält die ursprüngliche Frage und die neue Antwort.

In [ ]:
ergebnis = app.invoke({"frage": "Was ist LangGraph? Erkläre es in zwei kurzen Sätzen."})
print(ergebnis["antwort"])

## Kleines Experiment

Die Frage durch „Was ist ein Knoten in LangGraph? Ein Satz genügt.“ ersetzen und die letzte Zelle erneut ausführen.

**State = Daten. Knoten = Arbeitsschritt. Kanten = Reihenfolge.**

Jeder Durchlauf startet neu. Dieses Beispiel speichert keinen Gesprächsverlauf.

Grundlagen: [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) und [Gemini-Integration](https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai).